# Official settlement-source anchor diagnostic

## tl;dr

The current fair-value path uses Binance for both the current price and five-minute window open even though Polymarket explicitly settles these markets against Chainlink BTC/USD. Across `1,389` fresh, paired-book-valid candidate-interval states, changing only the spot/strike anchor to Chainlink moved fair probability by a median `2.02 pp` and `5.51 pp` at p90. It changed the fixed fee-aware `0.07` edge classification in `215 / 2,778` orientations (`7.74%`), spanning `21 / 24` markets and both directions. All `24 / 24` captured Chainlink opens matched Polymarket's published price to beat. This is the first label-free candidate with distinct, distributed decision selectivity. It warrants a disjoint preregistered evaluation, not a runtime change or profitability claim.


## Context & Methods

Polymarket's market rules state that five-minute BTC Up/Down contracts compare the Chainlink BTC/USD value at the end of the window with its value at the beginning, not another spot venue. The official RTDS documentation exposes Binance and Chainlink as separate sources. The current harness uses the proxy tape for `btc_price` and `open_btc`, while the separate settlement tape is used only for the terminal label.

This diagnostic changes exactly one modeling input: fair value uses the latest causal Chainlink price and the Chainlink price-to-beat strike. Binance remains the source for realized volatility and momentum. The Black–Scholes binary formula, volatility floor `0.30`, fee formula, ask prices, price band, and minimum edge `0.07` remain unchanged.

### Key Assumptions

- Population: the frozen 120–179 second interval in 24 non-overlapping five-minute BTC markets from the retained July 15 capture.
- Official current price must be no more than `10 s` old; the opening strike must be observed within `2 s` of the market open. Missing or stale official data fails closed.
- Each valid state is expanded to Up and Down orientations only to measure whether the anchor can change edge classification. These are not strategy candidates because confidence, z-score, state, and terminal outcomes are intentionally excluded.
- Polymarket page values are used only to verify that the captured Chainlink open equals the published price to beat; no resolution outcome is parsed.
- This is structural model-specification evidence, not a backtest, fill simulation, or profitability estimate.


## Data

### 1. Load the two source tapes and frozen market universe


In [1]:
import bisect
import csv
import gzip
import json
import math
import statistics
from collections import defaultdict
from datetime import datetime
from pathlib import Path


ROOT = Path("/Users/ttoomm/Documents/PolyMomentum")
CAPTURE = Path("/private/tmp/fresh-block-canary-recovered/segment_001")
CONVERTED = CAPTURE / "converted_v10"
RAW = CAPTURE / "raw"
manifest = json.loads((CONVERTED / "manifest.json").read_text())


def timestamp(value):
    return datetime.fromisoformat(value.replace("Z", "+00:00")).timestamp()


def load_tape(path):
    first_by_ts = {}
    with path.open() as handle:
        for row in csv.DictReader(handle):
            ts_ms = int(float(row["timestamp_ms"]))
            price = float(row["price"])
            if price > 0 and math.isfinite(price):
                first_by_ts.setdefault(ts_ms, price)
    pairs = sorted(first_by_ts.items())
    return [ts for ts, _ in pairs], [price for _, price in pairs]


def at(tape, timestamp_ms):
    timestamps, prices = tape
    index = bisect.bisect_right(timestamps, timestamp_ms) - 1
    if index < 0:
        return None, None
    return prices[index], timestamp_ms - timestamps[index]


def realized_vol(tape, timestamp_ms):
    timestamps, prices = tape
    lo = bisect.bisect_left(timestamps, timestamp_ms - 3_600_000)
    hi = bisect.bisect_right(timestamps, timestamp_ms)
    values = []
    total_dt = 0.0
    for index in range(lo + 1, hi):
        dt = (timestamps[index] - timestamps[index - 1]) / 1000
        if dt > 0:
            values.append(math.log(prices[index] / prices[index - 1]))
            total_dt += dt
    if len(values) < 30 or total_dt <= 0:
        return 0.5
    variance = statistics.pvariance(values)
    average_dt = total_dt / len(values)
    return min(max(math.sqrt(variance / average_dt * 365.25 * 86400), 0.05), 5.0)


def fair(spot, strike, minutes_remaining, sigma):
    years = (minutes_remaining / 1440) / 365.25
    d2 = (math.log(spot / strike) + (0.05 - 0.5 * sigma * sigma) * years) / (
        sigma * math.sqrt(years)
    )
    return min(max(0.5 * (1 + math.erf(d2 / math.sqrt(2))), 0.01), 0.99)


binance = load_tape(RAW / "binance_btcusdt_rtds.csv")
chainlink = load_tape(RAW / "chainlink_btcusd.csv")
windows = {}
tokens = defaultdict(dict)
for condition_id, market in manifest["markets"].items():
    close = timestamp(market["end_date"])
    windows[condition_id] = (close - 300, close)
    for outcome in market["outcomes"]:
        tokens[condition_id][outcome["name"].lower()] = outcome["token_id"]


### 2. Reconstruct causal paired books and both source anchors


In [2]:
books = defaultdict(lambda: {"snapshot": False, "bids": {}, "asks": {}, "bb": 0.0, "ba": 0.0, "ts": None})
last_effective = defaultdict(lambda: float("-inf"))
next_second = {condition_id: int(open_ts) for condition_id, (open_ts, _) in windows.items()}
rows = []


def book_metrics(book, cutoff):
    if not book["snapshot"] or book["ts"] is None or not 0 <= cutoff - book["ts"] <= 30:
        return None
    if not 0 < book["bb"] < book["ba"] < 1:
        return None
    bids = sorted(
        ((price, size) for price, size in book["bids"].items() if size > 0 and price <= book["bb"] + 1e-9),
        reverse=True,
    )[:3]
    asks = sorted(
        (price, size) for price, size in book["asks"].items() if size > 0 and price >= book["ba"] - 1e-9
    )[:3]
    if not bids or not asks or sum(size for _, size in bids) <= 0 or sum(size for _, size in asks) <= 0:
        return None
    return {"ask": book["ba"]}


def sample(condition_id, second):
    open_ts, _ = windows[condition_id]
    elapsed = second - int(open_ts)
    if not 120 <= elapsed < 180:
        return
    cutoff = second + 1 - 1e-9
    up = book_metrics(books[tokens[condition_id]["up"]], cutoff)
    down = book_metrics(books[tokens[condition_id]["down"]], cutoff)
    if up is None or down is None:
        return
    open_ms = int(open_ts * 1000)
    current_ms = second * 1000
    proxy_open, proxy_open_age = at(binance, open_ms)
    official_open, official_open_age = at(chainlink, open_ms)
    proxy_current, proxy_age = at(binance, current_ms)
    official_current, official_age = at(chainlink, current_ms)
    if None in (proxy_open, official_open, proxy_current, official_current):
        return
    sigma = max(realized_vol(binance, current_ms), 0.30)
    minutes_remaining = (300 - elapsed) / 60
    proxy_up = fair(proxy_current, proxy_open, minutes_remaining, sigma)
    official_up = fair(official_current, official_open, minutes_remaining, sigma)
    fee = lambda ask: 0.07 * ask * (1 - ask)
    proxy_edges = {
        "up": proxy_up - up["ask"] - fee(up["ask"]),
        "down": (1 - proxy_up) - down["ask"] - fee(down["ask"]),
    }
    official_edges = {
        "up": official_up - up["ask"] - fee(up["ask"]),
        "down": (1 - official_up) - down["ask"] - fee(down["ask"]),
    }
    rows.append({
        "condition_id": condition_id,
        "elapsed": elapsed,
        "proxy_open": proxy_open,
        "official_open": official_open,
        "proxy_current": proxy_current,
        "official_current": official_current,
        "proxy_age_ms": proxy_age,
        "official_age_ms": official_age,
        "official_open_age_ms": official_open_age,
        "proxy_move": proxy_current - proxy_open,
        "official_move": official_current - official_open,
        "sigma": sigma,
        "proxy_up": proxy_up,
        "official_up": official_up,
        "asks": {"up": up["ask"], "down": down["ask"]},
        "proxy_edges": proxy_edges,
        "official_edges": official_edges,
    })


for hour in manifest["hours"]:
    with gzip.open(hour["path"], "rt") as handle:
        for line in handle:
            event = json.loads(line)
            if event["ev"] == "trade" or event["mkt"] not in windows:
                continue
            condition_id = event["mkt"]
            effective = max(last_effective[condition_id], float(event["ts"]))
            last_effective[condition_id] = effective
            close = int(windows[condition_id][1])
            while next_second[condition_id] < close and next_second[condition_id] + 1 - 1e-9 < effective:
                sample(condition_id, next_second[condition_id])
                next_second[condition_id] += 1
            book = books[event["tok"]]
            if event["ev"] == "book":
                book["bids"] = {float(price): float(size) for price, size in event["bids"] if float(size) > 0}
                book["asks"] = {float(price): float(size) for price, size in event["asks"] if float(size) > 0}
                book["snapshot"] = True
            else:
                side = "bids" if event["s"] == "BUY" else "asks"
                price = float(event["p"])
                size = float(event["sz"])
                if size > 0:
                    book[side][price] = size
                else:
                    book[side].pop(price, None)
            book.update(bb=float(event["bb"]), ba=float(event["ba"]), ts=effective)

for condition_id, (_, close_ts) in windows.items():
    while next_second[condition_id] < int(close_ts):
        sample(condition_id, next_second[condition_id])
        next_second[condition_id] += 1


## Results

### 3. Hold every non-anchor input fixed and compare edge decisions


In [3]:
def dist(values):
    values = sorted(values)
    def q(p):
        if not values:
            return None
        x = (len(values) - 1) * p
        lo, hi = math.floor(x), math.ceil(x)
        return values[lo] if lo == hi else values[lo] * (hi - x) + values[hi] * (x - lo)
    return {"n": len(values), "min": min(values), "p50": q(.5), "p90": q(.9), "p99": q(.99), "max": max(values), "mean": statistics.fmean(values)}


orientation_rows = []
for row in rows:
    for direction in ("up", "down"):
        ask = row["asks"][direction]
        in_price_band = 0.1 <= ask <= 0.85
        proxy_pass = in_price_band and row["proxy_edges"][direction] >= 0.07
        official_pass = in_price_band and row["official_edges"][direction] >= 0.07
        orientation_rows.append((proxy_pass, official_pass, direction, row))

summary = {
    "states": len(rows),
    "conditions": len({row["condition_id"] for row in rows}),
    "official_current_age_ms": dist([row["official_age_ms"] for row in rows]),
    "official_age_over_10s": sum(row["official_age_ms"] > 10_000 for row in rows),
    "move_difference_usd": dist([row["official_move"] - row["proxy_move"] for row in rows]),
    "abs_move_difference_usd": dist([abs(row["official_move"] - row["proxy_move"]) for row in rows]),
    "direction_disagreements": sum((row["proxy_move"] >= 0) != (row["official_move"] >= 0) for row in rows),
    "fair_probability_delta": dist([row["official_up"] - row["proxy_up"] for row in rows]),
    "abs_fair_probability_delta": dist([abs(row["official_up"] - row["proxy_up"]) for row in rows]),
    "orientation_edge_disagreements": sum(proxy != official for proxy, official, _, _ in orientation_rows),
    "proxy_only_passes": sum(proxy and not official for proxy, official, _, _ in orientation_rows),
    "official_only_passes": sum(official and not proxy for proxy, official, _, _ in orientation_rows),
    "both_pass": sum(proxy and official for proxy, official, _, _ in orientation_rows),
    "neither_pass": sum(not proxy and not official for proxy, official, _, _ in orientation_rows),
}
fresh_rows = [
    row
    for row in rows
    if row["official_age_ms"] <= 10_000 and row["official_open_age_ms"] <= 2_000
]
fresh_ids = {(row["condition_id"], row["elapsed"]) for row in fresh_rows}
fresh_orientations = [
    entry
    for entry in orientation_rows
    if (entry[3]["condition_id"], entry[3]["elapsed"]) in fresh_ids
]
ordered_conditions = sorted(windows, key=lambda condition_id: windows[condition_id][0])
condition_half = {
    condition_id: "first" if index < len(ordered_conditions) / 2 else "second"
    for index, condition_id in enumerate(ordered_conditions)
}
summary["fresh_official_anchor"] = {
    "states": len(fresh_rows),
    "conditions": len({row["condition_id"] for row in fresh_rows}),
    "direction_disagreements": sum(
        (row["proxy_move"] >= 0) != (row["official_move"] >= 0)
        for row in fresh_rows
    ),
    "abs_move_difference_usd": dist(
        [abs(row["official_move"] - row["proxy_move"]) for row in fresh_rows]
    ),
    "abs_fair_probability_delta": dist(
        [abs(row["official_up"] - row["proxy_up"]) for row in fresh_rows]
    ),
    "orientation_edge_disagreements": sum(
        proxy != official for proxy, official, _, _ in fresh_orientations
    ),
    "proxy_only_passes": sum(
        proxy and not official for proxy, official, _, _ in fresh_orientations
    ),
    "official_only_passes": sum(
        official and not proxy for proxy, official, _, _ in fresh_orientations
    ),
    "both_pass": sum(proxy and official for proxy, official, _, _ in fresh_orientations),
    "neither_pass": sum(
        not proxy and not official for proxy, official, _, _ in fresh_orientations
    ),
    "edge_disagreements_by_direction": {
        direction: sum(
            proxy != official
            for proxy, official, row_direction, _ in fresh_orientations
            if row_direction == direction
        )
        for direction in ("up", "down")
    },
    "edge_disagreements_by_chronological_half": {
        half: sum(
            proxy != official
            for proxy, official, _, row in fresh_orientations
            if condition_half[row["condition_id"]] == half
        )
        for half in ("first", "second")
    },
    "conditions_with_edge_disagreement": len(
        {
            row["condition_id"]
            for proxy, official, _, row in fresh_orientations
            if proxy != official
        }
    ),
    "maximum_condition_edge_disagreements": max(
        sum(
            proxy != official
            for proxy, official, _, row in fresh_orientations
            if row["condition_id"] == condition_id
        )
        for condition_id in windows
    ),
}
print(json.dumps(summary, indent=2))


{
  "states": 1403,
  "conditions": 24,
  "official_current_age_ms": {
    "n": 1403,
    "min": 0,
    "p50": 0,
    "p90": 0.0,
    "p99": 9980.000000000018,
    "max": 24000,
    "mean": 217.3913043478261
  },
  "official_age_over_10s": 14,
  "move_difference_usd": {
    "n": 1403,
    "min": -15.606753802567255,
    "p50": 0.7198866640683264,
    "p90": 6.886595431319437,
    "p99": 11.313629478100784,
    "max": 43.71794659634179,
    "mean": 0.8127280983677293
  },
  "abs_move_difference_usd": {
    "n": 1403,
    "min": 0.00021490168728632852,
    "p50": 3.2165034332938376,
    "p90": 7.092705344407295,
    "p99": 12.160762260517668,
    "max": 43.71794659634179,
    "mean": 3.6804782315708935
  },
  "direction_disagreements": 34,
  "fair_probability_delta": {
    "n": 1403,
    "min": -0.10422107189303109,
    "p50": 0.0024225948516364837,
    "p90": 0.051591505652526765,
    "p99": 0.07739296196729394,
    "max": 0.23354981543852243,
    "mean": 0.005153609159652908
  },
  "ab

### 4. Verify the published strikes and save the evidence contract


In [4]:
import hashlib
import re
from datetime import timezone

EVIDENCE_PATH = ROOT / 'deploy/promotions/evidence/strategy_registry/20260721_settlement_source_anchor_diagnostic.json'
PRICE_MANIFEST_PATH = ROOT / 'deploy/promotions/evidence/strategy_registry/source_snapshots/20260721_settlement_anchor_price_to_beat_manifest.json'

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

price_rows = []
for page_path in sorted(Path('/private/tmp').glob('ptb_btc-updown-5m-*.html')):
    slug = page_path.stem.removeprefix('ptb_')
    open_ts_s = int(slug.rsplit('-', 1)[1])
    text = page_path.read_text()
    parsed_values = []
    for raw_value in re.findall(r'priceToBeat\\?":([0-9.]+)', text):
        value = float(raw_value)
        if value not in parsed_values:
            parsed_values.append(value)
    captured_open, captured_age_ms = at(chainlink, open_ts_s * 1000)
    assert captured_open is not None
    assert captured_age_ms <= 2_000
    published = min(parsed_values, key=lambda value: abs(value - captured_open))
    price_rows.append({
        'slug': slug,
        'url': f'https://polymarket.com/event/{slug}',
        'open_ts_s': open_ts_s,
        'published_price_to_beat': published,
        'captured_chainlink_open': captured_open,
        'absolute_difference_usd': abs(published - captured_open),
        'page_sha256': sha256_file(page_path),
    })

assert len(price_rows) == 24
assert max(row['absolute_difference_usd'] for row in price_rows) < 1e-9

generated_at = datetime.now(timezone.utc).isoformat()
price_manifest = {
    'schema_version': 1,
    'generated_at': generated_at,
    'status': 'PUBLIC_PRICE_TO_BEAT_MATCHES_CAPTURED_CHAINLINK_OPEN_24_OF_24',
    'selection': 'all 24 markets in the retained label-free source-anchor diagnostic',
    'source_page_contents_retained': False,
    'pages': price_rows,
}
temporary = PRICE_MANIFEST_PATH.with_name(f'{PRICE_MANIFEST_PATH.name}.tmp')
temporary.write_text(json.dumps(price_manifest, indent=2, sort_keys=True) + '\n')
temporary.replace(PRICE_MANIFEST_PATH)

fresh = summary['fresh_official_anchor']
fresh_orientation_count = fresh['both_pass'] + fresh['neither_pass'] + fresh['proxy_only_passes'] + fresh['official_only_passes']
assert fresh_orientation_count == 2 * fresh['states']
assert fresh['orientation_edge_disagreements'] == fresh['proxy_only_passes'] + fresh['official_only_passes']

source_hashes = {
    'binance_btcusdt_rtds.csv': sha256_file(RAW / 'binance_btcusdt_rtds.csv'),
    'chainlink_btcusd.csv': sha256_file(RAW / 'chainlink_btcusd.csv'),
    'manifest.json': sha256_file(CONVERTED / 'manifest.json'),
    'rust_engine/src/backtest/btc_history.rs': sha256_file(ROOT / 'rust_engine/src/backtest/btc_history.rs'),
    'rust_engine/src/backtest/harness.rs': sha256_file(ROOT / 'rust_engine/src/backtest/harness.rs'),
    'rust_engine/src/fair_value.rs': sha256_file(ROOT / 'rust_engine/src/fair_value.rs'),
    'rust_engine/src/strategy/decision.rs': sha256_file(ROOT / 'rust_engine/src/strategy/decision.rs'),
}
for hour in manifest['hours']:
    event_path = Path(hour['path'])
    source_hashes[event_path.name] = sha256_file(event_path)

evidence = {
    'schema_version': 1,
    'generated_at': generated_at,
    'mechanism_id': 'settlement_source_anchor_v1',
    'status': 'LABEL_FREE_SETTLEMENT_SOURCE_ANCHOR_SHOWS_DISTINCT_DECISION_SELECTIVITY_PREREGISTRATION_WARRANTED',
    'decision_question': 'Does anchoring fair value to the official Chainlink settlement process materially change fee-aware edge decisions while all other model inputs remain fixed?',
    'source_authority': {
        'capture': 'abandoned fresh-block-canary captured 2026-07-15T06:49Z through 2026-07-15T08:50Z',
        'capture_conditions': 24,
        'official_market_rule_example': 'https://polymarket.com/event/btc-updown-5m-1784098200',
        'official_rtds_documentation': 'https://docs.polymarket.com/market-data/websocket/rtds',
        'price_to_beat_manifest': str(PRICE_MANIFEST_PATH.relative_to(ROOT)),
        'source_sha256': source_hashes,
        'resolution_manifest_loaded': False,
        'terminal_labels_loaded': False,
        'strategy_outcomes_loaded': False,
        'active_forward_block_loaded': False,
    },
    'model_contract': {
        'baseline_anchor': 'Binance RTDS current price divided by Binance RTDS five-minute window open',
        'candidate_anchor': 'latest causal Chainlink RTDS current price divided by the published Chainlink price-to-beat window open',
        'unchanged_inputs': [
            'Binance one-hour realized volatility with floor 0.30',
            'Binance momentum and direction signal',
            'binary option formula and 0.05 annual risk-free rate',
            'paired executable asks',
            'Polymarket 0.07 crypto taker fee formula',
            'price band 0.10 through 0.85',
            'minimum fee-aware edge 0.07',
        ],
        'official_current_max_age_ms': 10_000,
        'official_open_max_age_ms': 2_000,
        'missing_or_stale_action': 'reject_row',
        'alternate_thresholds_tested': 0,
    },
    'data_quality': {
        'possible_candidate_condition_seconds': 24 * 60,
        'paired_book_valid_condition_seconds': len(rows),
        'fresh_official_anchor_condition_seconds': fresh['states'],
        'fresh_official_anchor_orientation_seconds': fresh_orientation_count,
        'book_invalid_condition_seconds': 24 * 60 - len(rows),
        'official_current_stale_over_10s_condition_seconds': summary['official_age_over_10s'],
        'published_price_to_beat_matches': len(price_rows),
        'published_price_to_beat_max_abs_difference_usd': max(row['absolute_difference_usd'] for row in price_rows),
        'promotion_or_exact_replay_eligible': False,
        'quality_assessment': 'SHARE_WITH_CAVEATS_FOR_LABEL_FREE_MODEL_SPECIFICATION_SCREEN_ONLY',
    },
    'structural_results': {
        'fresh_official_anchor': fresh,
        'full_paired_book_valid_population': summary,
        'edge_disagreement_rate': fresh['orientation_edge_disagreements'] / fresh_orientation_count,
        'proxy_only_pass_rate': fresh['proxy_only_passes'] / fresh_orientation_count,
        'official_only_pass_rate': fresh['official_only_passes'] / fresh_orientation_count,
        'published_price_to_beat_verification': {
            'markets': len(price_rows),
            'exact_matches_within_1e_9_usd': sum(row['absolute_difference_usd'] < 1e-9 for row in price_rows),
            'maximum_absolute_difference_usd': max(row['absolute_difference_usd'] for row in price_rows),
        },
    },
    'mechanism_assessment': {
        'distinct_decision_information_observed': True,
        'distributed_across_markets': fresh['conditions_with_edge_disagreement'] >= 12,
        'both_directions_affected': all(value > 0 for value in fresh['edge_disagreements_by_direction'].values()),
        'chronological_concentration_observed': fresh['edge_disagreements_by_chronological_half']['second'] > 4 * fresh['edge_disagreements_by_chronological_half']['first'],
        'interpretation': 'The official settlement anchor is not algebraically redundant with the proxy anchor and changes a material number of fee-aware edge decisions. The effect is broad across markets and directions but concentrated in the second chronological half, so profitability and stability remain unknown.',
        'why_preregistration_is_warranted': 'The candidate corrects source alignment to the contract being priced, all 24 official strikes reproduce exactly, and decision disagreements are nonzero without labels or threshold search.',
    },
    'decision': {
        'candidate_preregistration_warranted': True,
        'runtime_or_live_strategy_changed': False,
        'active_binary_complement_block_changed': False,
        'same_block_second_hypothesis_score_permitted': False,
        'required_next_evidence': 'a disjoint post-registration block under a frozen official-source freshness and edge contract, followed by exact measured-latency replay only if the screen passes',
        'a_plus_claim': False,
        'profitability_claim': False,
        'live_trading': 'OFF',
    },
    'limitations': [
        'The diagnostic contains no strategy confidence, z-score, state, terminal labels, fills, or realized PnL; orientation-level edge decisions are not trades.',
        'The source capture has only 24 markets, incomplete one-hour warm-up, reference-tape gaps, and 865 native CLOB timestamp regressions.',
        'Fourteen paired-book-valid seconds fail the candidate 10-second official-current freshness limit and are excluded.',
        'The edge disagreements are chronologically concentrated: 30 in the first 12 markets and 185 in the last 12.',
        'The candidate retains Binance volatility and momentum; it tests settlement anchoring, not a fully Chainlink-derived forecast process.',
        'Exact profitability still requires causal signal gates, measured latency, L2 fills, fees, stateful sizing, breadth, and tail tests.',
    ],
}

temporary = EVIDENCE_PATH.with_name(f'{EVIDENCE_PATH.name}.tmp')
temporary.write_text(json.dumps(evidence, indent=2, sort_keys=True) + '\n')
temporary.replace(EVIDENCE_PATH)

print({
    'artifact': str(EVIDENCE_PATH.relative_to(ROOT)),
    'price_manifest': str(PRICE_MANIFEST_PATH.relative_to(ROOT)),
    'fresh_states': fresh['states'],
    'edge_disagreements': fresh['orientation_edge_disagreements'],
    'edge_disagreement_rate': evidence['structural_results']['edge_disagreement_rate'],
    'markets_with_disagreement': fresh['conditions_with_edge_disagreement'],
    'price_to_beat_matches': len(price_rows),
    'status': evidence['status'],
})


{'artifact': 'deploy/promotions/evidence/strategy_registry/20260721_settlement_source_anchor_diagnostic.json', 'price_manifest': 'deploy/promotions/evidence/strategy_registry/source_snapshots/20260721_settlement_anchor_price_to_beat_manifest.json', 'fresh_states': 1389, 'edge_disagreements': 215, 'edge_disagreement_rate': 0.07739380849532038, 'markets_with_disagreement': 21, 'price_to_beat_matches': 24, 'status': 'LABEL_FREE_SETTLEMENT_SOURCE_ANCHOR_SHOWS_DISTINCT_DECISION_SELECTIVITY_PREREGISTRATION_WARRANTED'}


## Takeaways

- **This is a model-source correction, not another fitted threshold.** The contract being priced is defined by Chainlink, and the captured Chainlink open exactly reproduces every published strike in the sample.
- **The correction is decision-relevant.** It changes `7.74%` of fixed edge classifications after fail-closing stale official values, affects Up and Down, and appears in `21 / 24` markets.
- **It is not yet performance evidence.** The diagnostic deliberately excludes signal eligibility and terminal outcomes; it cannot say whether the changed decisions are better.
- **Do not score it on the active binary-complement block.** Freeze a disjoint evaluation contract now to avoid multiplying hypotheses on the same sealed outcomes. Only a fresh block may determine whether official-source alignment improves Wilson confidence, fee-inclusive economics, chronological stability, and loss clustering.
